# Alert 생성기 실험 ③ — 모델 없이 패턴 룰만 vs 이진분류만

같은 원장에서 **세탁 거래를 고르는 일**을 두 가지 방법으로 하고 정답 라벨로 잰다. 묶기(Alert 생성기)는 이 실험에 없다.

| 방법 | 고르는 기준 | 쓰는 것 |
|---|---|---|
| 룰만 | 패턴 6종을 룰로 적어 원장 전체에 건다. 룰에 걸린 근거 거래가 "고른 거래"다 | 거래 시각, 계좌, 금액. **모델 점수는 쓰지 않는다** |
| 이진분류만 | 점수 ≥ 의심 기준 | 이진 모델의 테스트 점수 |

- 룰 6종: FAN-OUT, FAN-IN, GATHER-SCATTER, SCATTER-GATHER, CYCLE, RANDOM. 정의는 3절과 `rule_lib.py` 에 있다. STACK·BIPARTITE 는 룰이 없다.
- 룰의 조건값(계좌 수, 경로 길이, 기간)은 미리 정하지 않는다. 후보를 전부 돌린 뒤 **보정 구간에서 고르고**(6절) **평가 구간에서 잰다**(7절). 모델이 검증 구간에서 의심 기준을 고르고 테스트 구간에서 재는 것과 같은 순서다.
- 룰은 패턴이 **완성된 뒤에야** 걸린다. 기본 설정(`LOOKAHEAD_DAYS = 0`)에서는 룰도 모델처럼 평가 구간 끝까지의 거래만 보므로, 구간 끝에 걸친 시도는 룰이 불리하다(1절에서 그런 시도 수를 센다).
- `MODEL = None` 이면 룰 쪽만 돈다. 8절과 9절의 모델 줄은 빠진다.
- 룰 SQL 은 HI-Small 에서 같은 정의를 Python 반복문으로 짠 계산과 결과가 같은 것을 확인했다(2026-11-23). HI-Large 로는 아직 안 돌려 봤다.

## 0. 설정

In [ ]:
DATASET = 'HI-Small'             # 'HI-Large' 는 아직 안 돌려 봤다

import json
from pathlib import Path

# --- 이진 모델 점수 (01번과 같은 방식). 아래 출력에 보이는 파일 중에서 MODEL 을 정한다
EXP = Path('../binary_classification/results') / DATASET / {'HI-Small': 'model_compare', 'HI-Large': 'gnn_structure'}[DATASET]
print(sorted(p.name for p in EXP.glob('*_test_scores.parquet')))
MODEL = None                     # 예: HI-Small 'gine_s0', HI-Large 'rev_port'. None 이면 룰 쪽만 돈다

In [ ]:
def load_threshold(exp, model):
    # 의심 기준은 그 모델이 검증 구간에서 고른 값을 쓴다 (테스트에서 고른 값을 쓰면 안 된다)
    one = exp / f'{model}_results.json'                 # 21번 방식: 모델마다 파일 하나
    if one.exists():
        return json.load(open(one, encoding='utf-8'))['val']['threshold']
    return json.load(open(exp / 'results.json', encoding='utf-8'))['results'][model]['val']['threshold']   # 09번 방식

SCORES = None if MODEL is None else EXP / f'{MODEL}_test_scores.parquet'
THRESHOLD = None if MODEL is None else load_threshold(EXP, MODEL)
print(SCORES, THRESHOLD)

In [ ]:
# --- 구간. 평가 구간 = 이진분류 실험의 테스트 구간. 보정 구간 = 그 앞 구간(룰 조건값을 고르는 데만 쓴다)
TEST_WINDOW  = {'HI-Small': ('2022-09-09', '2022-09-10'), 'HI-Large': ('2022-10-17', '2022-10-30')}[DATASET]
CALIB_WINDOW = {'HI-Small': ('2022-09-01', '2022-09-08'), 'HI-Large': ('2022-10-03', '2022-10-16')}[DATASET]
PAD_DAYS = 14                    # 보정 구간 앞으로 더 읽는 날짜 수. 룰이 돌아볼 거래다 (가장 긴 기간 이상)
LOOKAHEAD_DAYS = 0               # 평가 구간 뒤로 룰이 더 보는 날짜 수. 0 = 모델과 같은 정보(평가 구간 끝까지). 아래 설명 참고

# --- 룰 조건
WINDOWS = [1440, 4320, 10080]    # 기간 후보(분): 1일, 3일, 7일. 룰엔진 설계안의 기간 목록에 있는 값
GRID = {                         # 조건값 후보. 세기가 이 값 이상이면 적중
    'fan_out':        [2, 3, 4, 5, 6, 8, 10, 12, 14, 16],    # 서로 다른 수취 계좌 수
    'fan_in':         [2, 3, 4, 5, 6, 8, 10, 12, 14, 16],    # 서로 다른 송금 계좌 수
    'gather_scatter': [2, 3, 4, 5, 6, 8, 10, 12],            # 송금 계좌 수와 수취 계좌 수 (둘 다)
    'scatter_gather': [2, 3, 4, 5, 6, 8, 10, 12],            # 경유 계좌 수
    'cycle':          [2, 3, 4, 5, 6, 8, 10],                # 순환의 거래 수
    'random':         [2, 3, 4, 5, 6, 7, 8, 10],             # 경로의 거래 수
}
RATIO_MIN, RATIO_MAX = 0.8, 1.25 # 경로형(CYCLE·RANDOM): 이어지는 두 거래의 금액 비(뒤 ÷ 앞, USD). 생성기 정책(block-pin-3)의 금액 띠와 같은 값
MAX_HOPS = 12                    # 경로형: 따라가는 최대 거래 수
BUSY_MAX, BUSY_MINUTES = 512, 4320   # 바쁜 계좌: 72시간 안 같은 방향 거래가 512건 초과. 생성기 허브 기준과 같은 숫자. BUSY_MAX = None 이면 끈다
PICK_BY = 'F1'                   # 6절에서 조건값을 고르는 기준 열

THREADS, MEMORY = 4, '8GB'       # DuckDB 설정

**`LOOKAHEAD_DAYS` 에 대해.** 룰은 뒤 거래가 들어와 패턴이 완성되면 앞 거래까지 근거로 잡는다. 실제 운영에서도 며칠 뒤 분석에서 잡힌다.
이 값을 올리면 "며칠 뒤에라도 잡히는가"를 볼 수 있다. 다만 HI-Small 은 09-10 뒤로 정상 거래가 거의 없고 세탁 거래만 이어져서 올리면 안 된다(룰에 유리해진다). HI-Large 에서만 쓴다.

**금액 비 0.8 ~ 1.25 에 대해.** HI-Small 패턴 파일에서 CYCLE·RANDOM 시도의 이어지는 거래 금액 비(USD 환산)는 98% 가 이 범위 안이다(2026-11-23 확인).
통화가 계속 바뀌어도 USD 로 바꾸면 거의 유지된다. 이 조건이 없으면 경로 수가 폭발해서 원장 전체에 경로형 룰을 걸 수 없다.

## 1. 거래 읽기
보정 구간 앞 `PAD_DAYS` 일부터 평가 구간 끝까지를 DuckDB 표로 올린다. 금액은 서비스와 같은 고정 환율로 USD 로 바꾼다.

In [ ]:
import time
import duckdb, numpy as np, pandas as pd
import gen_lib as GL
import rule_lib as RL

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'parquets').is_dir()), None)
assert ROOT is not None, 'parquets/ 폴더를 못 찾음. 데이터가 있는 GPU 서버의 저장소 안에서 열어야 한다'
BASE = ROOT / 'parquets' / DATASET
OUT = Path.cwd() / 'results' / DATASET / 'rule_vs_model'
OUT.mkdir(parents=True, exist_ok=True)
START = str((pd.Timestamp(CALIB_WINDOW[0]) - pd.Timedelta(days=PAD_DAYS)).date())
END = str((pd.Timestamp(TEST_WINDOW[1]) + pd.Timedelta(days=LOOKAHEAD_DAYS)).date())
con = duckdb.connect()
con.execute(f'SET threads={THREADS}'); con.execute(f"SET memory_limit='{MEMORY}'")
print(BASE, '| 읽는 기간', START, '~', END, '| 보정', CALIB_WINDOW, '| 평가', TEST_WINDOW, '| duckdb', duckdb.__version__)

In [ ]:
RL.load_tx(con, BASE, START, END, GL.UNITS_PER_USD)     # 환율_모르는_거래 는 0 이어야 정상

In [ ]:
# 구간별 세탁 거래 수 (패턴별). 평가 구간의 수가 작으면 비율이 크게 흔들린다
IN_CALIB = f"day BETWEEN DATE '{CALIB_WINDOW[0]}' AND DATE '{CALIB_WINDOW[1]}'"
IN_TEST = f"day BETWEEN DATE '{TEST_WINDOW[0]}' AND DATE '{TEST_WINDOW[1]}'"
con.execute(f'''SELECT CASE WHEN y = 1 THEN pattern_type ELSE '(정상)' END AS 패턴,
                       count(*) FILTER (WHERE {IN_CALIB}) AS 보정_거래, count(DISTINCT attempt_id) FILTER (WHERE {IN_CALIB}) AS 보정_시도,
                       count(*) FILTER (WHERE {IN_TEST}) AS 평가_거래, count(DISTINCT attempt_id) FILTER (WHERE {IN_TEST}) AS 평가_시도
                FROM tx_all GROUP BY 1 ORDER BY 1''').df()

In [ ]:
# 평가 구간에 걸친 시도 중, 구간이 끝난 뒤에도 거래가 이어지는 시도. 룰이 볼 수 있는 모양이 덜 완성돼 있다
con.execute(f'''WITH a AS (SELECT attempt_id, any_value(pattern_type) AS 패턴, max(ts) AS last_ts,
                                 bool_or(CAST(ts AS DATE) BETWEEN DATE '{TEST_WINDOW[0]}' AND DATE '{TEST_WINDOW[1]}') AS touch
                          FROM read_parquet('{BASE / 'patterns.parquet'}') GROUP BY attempt_id)
                SELECT 패턴, count(*) AS 평가_시도, count(*) FILTER (WHERE CAST(last_ts AS DATE) > DATE '{TEST_WINDOW[1]}') AS 구간_뒤로_이어짐,
                       count(*) FILTER (WHERE CAST(last_ts AS DATE) > DATE '{END}') AS 룰이_끝을_못_봄
                FROM a WHERE touch GROUP BY 1 ORDER BY 1''').df()

## 2. 바쁜 계좌
거래가 몰리는 계좌(정산·수납 계좌 등)는 "상대 계좌가 많다"가 정보가 아니고, 경로가 그 계좌를 지나면 아무 거래나 이어진다.
그래서 바쁜 계좌는 **룰의 대상도 경유도 되지 않는다**(상대 계좌로는 센다). 룰엔진 설계안 3.6 을 단순하게 옮긴 것이다.
바쁜 계좌에 닿은 세탁 거래는 룰이 그 계좌 쪽에서는 못 잡는다. 그 수를 여기서 확인한다.

In [ ]:
RL.busy_accounts(con, BUSY_MAX, BUSY_MINUTES)

## 3. 룰 돌리기
모든 룰이 같은 방식으로 돈다.

1. **트리거**: 대상 계좌의 거래 하나하나를 검사 시점으로 삼는다.
2. **기간**: 트리거 시각을 포함해 직전 `W` 분을 돌아본다.
3. **세기**: 그 기간 안에서 패턴의 크기(계좌 수, 경로 길이)를 센다. 세기가 조건값 이상이면 적중이다.
4. **근거 거래**: 적중한 트리거의 기간 안에서 그 패턴을 이루는 거래. 이것이 룰이 "고른 거래"다.

거래마다 "자기를 근거로 삼는 트리거들의 세기 중 최댓값"을 저장한다. 그러면 조건값을 바꿀 때 룰을 다시 돌리지 않고 `세기 ≥ 조건값` 으로 걸러 보기만 하면 된다.

### 3-1. FAN-OUT · FAN-IN (집계형)

| | FAN-OUT | FAN-IN |
|---|---|---|
| 모양 | A → 여러 계좌 | 여러 계좌 → A |
| 대상 | 송금 계좌 A | 수취 계좌 A |
| 트리거 | A 의 출금 | A 로 들어온 입금 |
| 세기 | 기간 안 **서로 다른 수취 계좌 수** | 기간 안 **서로 다른 송금 계좌 수** |
| 근거 거래 | 기간 안 A 의 출금 전부 | 기간 안 A 의 입금 전부 |

금액 조건은 없다. 같은 계좌로 여러 번 보낸 것은 계좌 하나로 센다.

In [ ]:
pd.DataFrame([RL.fan(con, side, W) for W in WINDOWS for side in ('out', 'in')])

### 3-2. GATHER-SCATTER (집계형)
여러 계좌 → H → 여러 계좌. 대상은 가운데 계좌 H, 트리거는 H 의 출금이다.

- `din` = 기간 안 입금의 서로 다른 송금 계좌 수
- `dout` = **기간 안 첫 입금 뒤** 출금의 서로 다른 수취 계좌 수 (모은 뒤에 뿌린 것만 센다)
- 세기 = `min(din, dout)`. 조건값 K 는 "양쪽 다 K 개 이상"이라는 뜻이다
- 근거 거래 = 기간 안 입금 전부 + 첫 입금 뒤의 출금

In [ ]:
pd.DataFrame([RL.gather_scatter(con, W) for W in WINDOWS])

### 3-3. SCATTER-GATHER (2홉형)
S → 경유 계좌 여러 개 → D. 대상은 도착 계좌 D, 트리거는 D 로 들어온 입금(경유 → D)이다.

- 세기 = 같은 출발 계좌 S 에서 와서 D 에 닿은 **경유 계좌 수**. 경유 계좌마다 `S → 경유` 가 `경유 → D` 보다 먼저(같은 분 포함)여야 하고 둘 다 기간 안이어야 한다
- 근거 거래 = 그 경유 계좌들의 `S → 경유` 거래와 `경유 → D` 거래
- S = D 는 순환이라 뺀다. 금액 조건은 없다

먼저 거래 시각과 무관하게 "경유 계좌를 2개 이상 공유하는 (S, D)"만 추리고, 그 안에서만 시각을 본다.

In [ ]:
RL.sg_candidates(con)

In [ ]:
pd.DataFrame([RL.scatter_gather(con, W) for W in WINDOWS])

### 3-4. CYCLE · RANDOM (경로형)
거래를 하나씩 이어 경로를 만든다. 두 거래가 **이어진다**는 것은 다음을 모두 만족한다는 뜻이다.

- 앞 거래의 수취 계좌 = 뒤 거래의 송금 계좌
- 뒤 거래가 앞 거래 이후(같은 분 포함)이고 기간 `W` 안
- 뒤 금액 ÷ 앞 금액(USD)이 `RATIO_MIN ~ RATIO_MAX`

경로는 같은 계좌를 두 번 지나지 않고, 첫 거래부터 마지막 거래까지가 `W` 안이며, 거래 수는 `MAX_HOPS` 까지다.

| | CYCLE | RANDOM |
|---|---|---|
| 모양 | A → B → … → A | A → B → C → … (돌아오지 않음) |
| 적중 | 경로가 **출발 계좌로 돌아옴** (거래 2건 이상) | 돌아오지 않은 경로 |
| 세기 | 그 거래가 든 순환 중 가장 긴 것의 거래 수 | 그 거래가 든 경로 중 가장 긴 것의 거래 수 |
| 근거 거래 | 순환의 거래 전부 | 경로의 거래 전부 |

순환의 앞부분도 닫히기 전까지는 "돌아오지 않은 경로"라서 RANDOM 에도 걸린다.

In [ ]:
pd.DataFrame([RL.links(con, W, RATIO_MIN, RATIO_MAX) for W in WINDOWS])

In [ ]:
# 경로를 만들기 전에 길이(거래 수)별 경로 수의 상한을 본다. 합이 수천만을 넘으면 다음 셀이 오래 걸리거나 메모리가 모자란다
pd.DataFrame({f'{W}분': RL.walk_counts(con, W, MAX_HOPS) for W in WINDOWS}).rename_axis('거래 수')

In [ ]:
pd.DataFrame([dict(기간_분=W, **RL.paths(con, W, MAX_HOPS)) for W in WINDOWS])

## 4. 결과 모으기
보정·평가 구간의 거래마다 룰별 세기(`<룰>_<기간>` 열)를 붙여 가져온다. 세기가 0 이면 그 룰에 안 걸린 거래다.

In [ ]:
F = RL.collect(con, WINDOWS, CALIB_WINDOW[0], TEST_WINDOW[1])
calib, test = RL.day_mask(F, *CALIB_WINDOW), RL.day_mask(F, *TEST_WINDOW)
print(f'{len(F):,} 건 | 보정 {calib.sum():,} | 평가 {test.sum():,} | 메모리 {F.memory_usage(deep=True).sum() / 1e6:.0f} MB')
print(list(F.columns))

## 5. 조건값을 바꾸면 (보정 구간)
룰 × 기간 × 조건값을 전부 걸어 본다. 아래 표는 룰 하나씩, 줄이 조건값이고 칸이 기간(분)이다.

- `대상_계좌`: 근거 거래가 걸린 서로 다른 대상 계좌 수. 사람이 봐야 할 양이다
- `고른_거래`: 룰이 고른 거래(근거 거래) 수
- `정밀도`: 고른 거래 중 세탁 거래 비율. 패턴 종류는 따지지 않는다 (FAN-OUT 룰이 GATHER-SCATTER 의 뿌리는 쪽을 잡아도 맞은 것으로 센다)
- `재현율`: **그 룰이 노리는 패턴**의 세탁 거래 중 고른 비율

In [ ]:
sweep_calib = RL.subjects(con, RL.sweep(F, GRID, WINDOWS, calib), *CALIB_WINDOW)
SHOW = ('대상_계좌', '고른_거래', '정밀도', '재현율')
RL.view(sweep_calib, 'fan_out', SHOW)

In [ ]:
RL.view(sweep_calib, 'fan_in', SHOW)

In [ ]:
RL.view(sweep_calib, 'gather_scatter', SHOW)

In [ ]:
RL.view(sweep_calib, 'scatter_gather', SHOW)

In [ ]:
RL.view(sweep_calib, 'cycle', SHOW)

In [ ]:
RL.view(sweep_calib, 'random', SHOW)

## 6. 조건값 고르기
룰마다 보정 구간에서 `PICK_BY`(기본 F1 = 위 정밀도와 재현율의 조화평균)가 가장 큰 (기간, 조건값)을 고른다.
모델의 의심 기준도 검증 구간에서 F1 이 가장 큰 값으로 고른 것이라 같은 기준이다.
다른 값을 쓰려면 아래 셀에서 `CHOSEN` 을 직접 바꾼다.

In [ ]:
CHOSEN = RL.pick(sweep_calib, PICK_BY)       # {룰: (기간_분, 조건값)}
# CHOSEN['fan_in'] = (4320, 5)               # 직접 정할 때
CHOSEN

In [ ]:
key = pd.DataFrame([dict(룰=RL.RULES[rule], 기간_분=W, 조건값=value) for rule, (W, value) in CHOSEN.items()])
key.merge(sweep_calib, on=['룰', '기간_분', '조건값'])      # 고른 조건의 보정 구간 결과

## 7. 평가 구간 — 룰만
고른 조건을 평가 구간에 그대로 건다. 여기부터는 조건값을 바꾸지 않는다.

In [ ]:
sweep_test = RL.subjects(con, RL.sweep(F, GRID, WINDOWS, test), *TEST_WINDOW)
rule_table = key.merge(sweep_test, on=['룰', '기간_분', '조건값'])
rule_table

In [ ]:
# 어떤 룰이 어떤 패턴의 세탁 거래를 잡았나. 값 = 그 패턴의 세탁 거래 중 그 룰이 고른 비율
each = {RL.RULES[rule]: RL.hit_mask(F, rule, W, value) for rule, (W, value) in CHOSEN.items()}
rule_hit = RL.union_mask(F, CHOSEN)          # 룰 6종 중 하나라도 고른 거래
cross = RL.by_pattern(F, test, {**each, '룰_6종_합집합': rule_hit})
cross

In [ ]:
# 룰 6종을 합친 결과. 보정 구간은 조건값을 고른 구간이라 조금 낙관적이지만, 룰이 뒤 거래까지 본 상태의 값이다
union_table = pd.DataFrame([RL.method_metrics('룰 6종 합집합 · 보정 구간', rule_hit, F, calib),
                            RL.method_metrics('룰 6종 합집합 · 평가 구간', rule_hit, F, test)])
union_table

## 8. 평가 구간 — 이진분류만

In [ ]:
if SCORES is None:
    model_hit = None
    print('MODEL 을 정하지 않아 모델 쪽은 건너뛴다. 0절에서 MODEL 을 정하고 처음부터 다시 돌린다')
else:
    scores = pd.read_parquet(SCORES)[['edge_id', 'y', 'score']].drop_duplicates('edge_id')     # edge_id, y, score
    joined = F[['edge_id', 'y']].merge(scores, on='edge_id', how='left', suffixes=('', '_score'))
    assert len(joined) == len(F)
    score = joined.score.to_numpy(dtype='float64')
    print(f'점수 {len(scores):,} 건 | 평가 구간인데 점수가 없는 거래 {int((test & np.isnan(score)).sum()):,} (0 이어야 정상)',
          '| 라벨 불일치', int((joined.y_score.notna() & (joined.y_score != joined.y)).sum()))
    model_hit = test & (np.nan_to_num(score, nan=-1.0) >= THRESHOLD)
    print('의심 기준', THRESHOLD, '| 모델이 고른 거래', int(model_hit.sum()))

## 9. 비교
### 9-1. 거래 단위
- `재현율_세탁_전체`: 평가 구간의 세탁 거래 전부가 분모 (무패턴 포함)
- `재현율_패턴_전체`: 패턴 파일에 있는 세탁만 분모 (STACK·BIPARTITE 포함)
- `재현율_룰이_있는_6종`: 룰을 만든 6개 패턴의 세탁만 분모
- `이진분류 상위 N건`: 룰이 고른 건수와 **같은 건수**를 모델 점수 높은 순으로 고른 것. 같은 검토량에서의 비교다

In [ ]:
methods = {'룰 6종 합집합': rule_hit & test}
single = {f'룰 · {name}': mask & test for name, mask in each.items()}      # 룰 하나씩 (참고 줄)
if model_hit is not None:
    n_rule = int((rule_hit & test).sum())
    order = np.argsort(-np.where(test, np.nan_to_num(score, nan=-1.0), -np.inf), kind='stable')
    top = np.zeros(len(F), dtype=bool)
    top[order[:n_rule]] = True
    methods = {'이진분류만': model_hit, **methods, f'이진분류 상위 {n_rule:,}건': top & test,
               '둘 다 고름': model_hit & rule_hit, '둘 중 하나라도': model_hit | (rule_hit & test)}
compare = pd.DataFrame([RL.method_metrics(name, mask, F, test) for name, mask in {**methods, **single}.items()])
compare

### 9-2. 패턴별 재현율

In [ ]:
pattern_table = RL.by_pattern(F, test, methods)
pattern_table

### 9-3. 세탁 거래를 누가 잡았나
평가 구간의 세탁 거래를 패턴별로 '둘 다 / 모델만 / 룰만 / 둘 다 놓침' 으로 나눈다. `룰만` 이 룰이 모델에 더해 주는 몫이다.

In [ ]:
if model_hit is None:
    who_table = None
    print('모델 점수가 없어 건너뛴다')
else:
    y, kind = F.y.to_numpy(), F.pattern_type.to_numpy()
    who = np.select([model_hit & rule_hit, model_hit, rule_hit], ['둘 다', '모델만', '룰만'], '둘 다 놓침')
    base = test & (y == 1)
    who_table = pd.crosstab(pd.Series(kind[base], name='패턴'), pd.Series(who[base], name='잡은 쪽'), margins=True)
who_table

### 9-4. 세탁 시도 단위
시도(Patterns 파일의 BEGIN ~ END 한 덩어리)마다 평가 구간 안 거래 중 몇 건을 골랐는지 본다. 값은 그런 시도의 비율이다.

In [ ]:
main = {name: mask for name, mask in methods.items() if name in ('이진분류만', '룰 6종 합집합')}
attempt_table = RL.by_attempt(F, test, main)
attempt_table

## 10. 저장

In [ ]:
sweep_calib.to_csv(OUT / 'sweep_calib.csv', index=False)
sweep_test.to_csv(OUT / 'sweep_test.csv', index=False)
rule_table.to_csv(OUT / 'rules.csv', index=False)
cross.to_csv(OUT / 'rule_by_pattern.csv', index=False)
union_table.to_csv(OUT / 'rule_union.csv', index=False)
compare.to_csv(OUT / 'compare.csv', index=False)
pattern_table.to_csv(OUT / 'by_pattern.csv', index=False)
attempt_table.to_csv(OUT / 'by_attempt.csv')
if who_table is not None:
    who_table.to_csv(OUT / 'who.csv')
# 고른 거래 목록. 룰 씨앗을 생성기에 넣는 다음 실험에서 읽는다
keep = (calib | test) & (rule_hit | (F.y.to_numpy() == 1) | (model_hit if model_hit is not None else False))
hits = F.loc[keep, ['edge_id', 'day', 'y', 'pattern_type', 'attempt_id']].copy()
for name, mask in each.items():
    hits[name] = mask[keep]
if model_hit is not None:
    hits['score'], hits['model'] = score[keep], model_hit[keep]
hits.to_parquet(OUT / 'hits.parquet', index=False)
json.dump(dict(dataset=DATASET, model=MODEL, scores=None if SCORES is None else str(SCORES), threshold=THRESHOLD,
               test_window=TEST_WINDOW, calib_window=CALIB_WINDOW, pad_days=PAD_DAYS, windows=WINDOWS, grid=GRID,
               ratio=[RATIO_MIN, RATIO_MAX], max_hops=MAX_HOPS, busy=[BUSY_MAX, BUSY_MINUTES], pick_by=PICK_BY,
               chosen={rule: list(value) for rule, value in CHOSEN.items()}, duckdb=duckdb.__version__),
          open(OUT / 'run.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print(sorted(p.name for p in OUT.iterdir()))